# 03 – Clean the data
Applies the decisions from `02_data_quality.ipynb` and saves a clean, analysis-ready file for SQL and Power BI.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/complaints_combined.csv", low_memory=False)
start_rows = len(df)
print(f"Starting rows: {start_rows:,}")

Starting rows: 569,138


## 1. Remove out-of-scope rows
- Credit bureaus (Equifax, TransUnion, Experian) – not banks
- Rows with no sub-product/issue, or the odd "Credit reporting" sub-product

In [2]:
bureaus = df["Company"].str.contains("EQUIFAX|TRANSUNION|EXPERIAN", case=False)
odd = df["Sub-product"].isna() | df["Issue"].isna() | (df["Sub-product"] == "Credit reporting")

print(f"Credit bureau rows removed: {bureaus.sum():,}")
print(f"Odd rows removed:           {(odd & ~bureaus).sum():,}")

df = df[~bureaus & ~odd].copy()
print(f"Rows left: {len(df):,}")

Credit bureau rows removed: 57,737
Odd rows removed:           6
Rows left: 511,395


## 2. Rename columns to snake_case
Short, lowercase names are easier to use in SQL and Power BI.

In [3]:
df = df.rename(columns={
    "Date received": "date_received",
    "Product": "product",
    "Sub-product": "sub_product",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "Company public response": "company_public_response",
    "Company": "company",
    "State": "state",
    "Tags": "tags",
    "Submitted via": "submitted_via",
    "Date sent to company": "date_sent",
    "Company response to consumer": "company_response",
    "Timely response?": "timely_response",
    "Complaint ID": "complaint_id",
})
df = df.drop(columns=["ZIP code", "source_file"], errors="ignore")
df.columns.tolist()

['date_received',
 'product',
 'sub_product',
 'issue',
 'sub_issue',
 'company_public_response',
 'company',
 'state',
 'tags',
 'submitted_via',
 'date_sent',
 'company_response',
 'timely_response',
 'complaint_id']

## 3. Dates
Convert to plain dates (no time) and add helper columns.

In [4]:
df["date_received"] = pd.to_datetime(df["date_received"], utc=True).dt.date
df["date_sent"] = pd.to_datetime(df["date_sent"], utc=True).dt.date

df["date_received"] = pd.to_datetime(df["date_received"])
df["date_sent"] = pd.to_datetime(df["date_sent"])

df["year"] = df["date_received"].dt.year
df["year_month"] = df["date_received"].dt.strftime("%Y-%m")
df["days_to_company"] = (df["date_sent"] - df["date_received"]).dt.days

df[["date_received", "date_sent", "year", "year_month", "days_to_company"]].head()

,date_received,date_sent,year,year_month,days_to_company
0,2023-03-11,2023-03-11,2023,2023-03,0
1,2023-03-11,2023-03-11,2023,2023-03,0
2,2023-03-11,2023-03-11,2023,2023-03,0
3,2023-03-11,2023-03-11,2023,2023-03,0
4,2023-03-11,2023-03-11,2023,2023-03,0


## 4. Fill blanks and group states

In [5]:
df["tags"] = df["tags"].fillna("No tag")
df["company_public_response"] = df["company_public_response"].fillna("No public response")
df["sub_issue"] = df["sub_issue"].fillna("Not provided")
df["state"] = df["state"].replace("UNITED STATES MINOR OUTLYING ISLANDS", "UM").fillna("Unknown")

us_50_dc = ["AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME",
            "MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI",
            "SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"]

def state_group(s):
    if s in us_50_dc:
        return "US state/DC"
    if s == "Unknown":
        return "Unknown"
    return "Military/Territory"

df["state_group"] = df["state"].apply(state_group)
print(df["state_group"].value_counts())

state_group
US state/DC           506102
Unknown                 3726
Military/Territory      1567
Name: count, dtype: int64


## 5. Outcome flags
These make the KPIs simple to calculate later:
- `is_closed` – the complaint has a final outcome (not "In progress")
- `got_relief` – customer got monetary or non-monetary relief
- `got_monetary_relief` – customer got money back
- `is_timely` – the company responded on time

In [6]:
df["is_closed"] = df["company_response"] != "In progress"
df["got_relief"] = df["company_response"].isin(["Closed with monetary relief", "Closed with non-monetary relief"])
df["got_monetary_relief"] = df["company_response"] == "Closed with monetary relief"
df["is_timely"] = df["timely_response"] == "Yes"

closed = df[df["is_closed"]]
print(f"Closed complaints: {len(closed):,}")
print(f"Relief rate (closed only):          {closed['got_relief'].mean():.1%}")
print(f"Monetary relief rate (closed only): {closed['got_monetary_relief'].mean():.1%}")
print(f"Timely response rate:               {df['is_timely'].mean():.1%}")

Closed complaints: 503,280
Relief rate (closed only):          23.7%
Monetary relief rate (closed only): 15.2%
Timely response rate:               99.3%


## 6. Final checks and save

In [7]:
assert df["complaint_id"].is_unique
assert df[["date_received", "product", "sub_product", "issue", "company", "state", "company_response"]].notna().all().all()

print(f"Rows: {start_rows:,} -> {len(df):,}  (removed {start_rows - len(df):,})")
print(f"Columns: {df.shape[1]}")
print(df["product"].value_counts())

df.to_csv("../data/processed/complaints_clean.csv", index=False)
print("\nSaved: ../data/processed/complaints_clean.csv")

Rows: 569,138 -> 511,395  (removed 57,743)
Columns: 22
product
Checking or savings account    264653
Credit card                    246742
Name: count, dtype: int64

Saved: ../data/processed/complaints_clean.csv
